# Intermediate: Transfer Learning

Master the art of using pretrained models to solve your problems with less data and training time!

Topics covered:
- What is transfer learning and why it works
- Using pretrained models (ResNet, VGG, EfficientNet)
- Feature extraction vs fine-tuning
- Domain adaptation
- Best practices for transfer learning
- Building custom classifiers on top of pretrained models

## Why Transfer Learning?

```
Traditional Training:        Transfer Learning:
Random → Train → Model      Pretrained → Fine-tune → Model
(needs lots of data)        (works with less data)
(trains from scratch)       (leverages learned features)

ImageNet (1M images)
    ↓
Pretrained Model
    ↓
Your Dataset (1K images) → Fine-tuned Model ✨
```

**Benefits:**
- 🚀 Faster training
- 📊 Better performance with less data
- 💡 Leverage knowledge from large datasets
- 🎯 Higher accuracy on small datasets


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import torchvision
from torchvision import datasets, models, transforms
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

print(f"PyTorch version: {torch.__version__}")
print(f"Torchvision version: {torchvision.__version__}")

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")


## 1. Loading Pretrained Models

PyTorch provides many pretrained models through `torchvision.models`.


In [ ]:
# Available pretrained models
print("Popular pretrained models:")
print("="*50)
print("ResNet family: resnet18, resnet34, resnet50, resnet101, resnet152")
print("VGG family: vgg11, vgg13, vgg16, vgg19")
print("DenseNet: densenet121, densenet169, densenet201")
print("EfficientNet: efficientnet_b0 to efficientnet_b7")
print("MobileNet: mobilenet_v2, mobilenet_v3_small, mobilenet_v3_large")
print("Vision Transformer: vit_b_16, vit_b_32, vit_l_16")
print()

# Load a pretrained ResNet18
model = models.resnet18(pretrained=True)
print("ResNet18 architecture:")
print(model)
print()

# Check the final layer
print(f"Original final layer (for ImageNet 1000 classes):")
print(f"  {model.fc}")
print(f"  Input features: {model.fc.in_features}")
print(f"  Output classes: {model.fc.out_features}")


## 2. Understanding the Model Architecture

Let's visualize what we're working with:

```
ResNet18 Architecture:

Input (3x224x224)
    ↓
Conv1 + BN + ReLU + MaxPool
    ↓
Layer1 (BasicBlock × 2)
    ↓
Layer2 (BasicBlock × 2)
    ↓
Layer3 (BasicBlock × 2)
    ↓
Layer4 (BasicBlock × 2)
    ↓
AvgPool
    ↓
FC (512 → 1000)  ← We'll modify this!
    ↓
Output (1000 classes)
```


In [ ]:
# Inspect model structure
print("Model children (main blocks):")
for name, child in model.named_children():
    print(f"  {name}: {type(child).__name__}")
print()

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")


## 3. Feature Extraction (Freeze Base Model)

Use pretrained model as fixed feature extractor.

```
Strategy 1: Feature Extraction

Pretrained Layers (FROZEN ❄️)
    ↓
[Conv blocks are frozen]
    ↓
New Classifier (TRAINABLE 🔥)
    ↓
Your Classes

Only train the new classifier!
```


In [ ]:
# Load pretrained ResNet18
model = models.resnet18(pretrained=True)

# Freeze all layers
for param in model.parameters():
    param.requires_grad = False

# Replace final layer for your task (e.g., 10 classes)
num_classes = 10
model.fc = nn.Linear(model.fc.in_features, num_classes)

# Move to device
model = model.to(device)

print("Feature Extraction Setup:")
print("="*50)

# Count trainable parameters now
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Frozen parameters: {total_params - trainable_params:,}")
print(f"Training only {100*trainable_params/total_params:.2f}% of parameters!")
print()

# Verify which layers are trainable
print("Trainable layers:")
for name, param in model.named_parameters():
    if param.requires_grad:
        print(f"  {name}: {param.shape}")


## 4. Fine-Tuning (Unfreeze and Train)

Fine-tune pretrained weights for your specific task.

```
Strategy 2: Fine-Tuning

Pretrained Layers (TRAINABLE 🔥)
    ↓
[Conv blocks can update]
    ↓
New Classifier (TRAINABLE 🔥)
    ↓
Your Classes

Train everything (usually with lower LR for pretrained parts)
```


In [ ]:
# Load pretrained model
model = models.resnet18(pretrained=True)

# Replace final layer
num_classes = 10
model.fc = nn.Linear(model.fc.in_features, num_classes)
model = model.to(device)

# Fine-tuning: Different learning rates for different parts
print("Fine-Tuning Setup:")
print("="*50)

# Lower LR for pretrained layers, higher LR for new classifier
optimizer = optim.SGD([
    {'params': model.layer1.parameters(), 'lr': 1e-4},
    {'params': model.layer2.parameters(), 'lr': 1e-4},
    {'params': model.layer3.parameters(), 'lr': 1e-4},
    {'params': model.layer4.parameters(), 'lr': 1e-4},
    {'params': model.fc.parameters(), 'lr': 1e-3}  # 10x higher for new layer
], momentum=0.9)

print("Learning rates:")
for i, param_group in enumerate(optimizer.param_groups):
    print(f"  Group {i}: lr = {param_group['lr']}")

# Alternative: Use all parameters with same LR
# optimizer = optim.SGD(model.parameters(), lr=1e-3)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nTraining {trainable_params:,} / {total_params:,} parameters (100%)")


## 5. Data Preprocessing for Pretrained Models

**Important:** Pretrained models expect specific input preprocessing!

```
ImageNet Normalization:
  mean = [0.485, 0.456, 0.406]
  std = [0.229, 0.224, 0.225]

Input size:
  ResNet, VGG: 224×224
  EfficientNet-B0: 224×224
  EfficientNet-B7: 600×600
```


In [ ]:
# Standard preprocessing for ImageNet models
train_transform = transforms.Compose([
    transforms.RandomResizedCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                        std=[0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                        std=[0.229, 0.224, 0.225])
])

print("Transforms created!")
print()
print("Training transforms (with augmentation):")
print(train_transform)
print()
print("Validation transforms (no augmentation):")
print(val_transform)


## 6. Complete Transfer Learning Pipeline

Let's put it all together with a real example.


In [ ]:
# Prepare data (using CIFAR-10 as example)
train_dataset = datasets.CIFAR10(
    './data', train=True, download=True, transform=train_transform
)
val_dataset = datasets.CIFAR10(
    './data', train=False, transform=val_transform
)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False, num_workers=2)

print(f"Training samples: {len(train_dataset)}")
print(f"Validation samples: {len(val_dataset)}")
print(f"Classes: {train_dataset.classes}")


In [ ]:
# Setup model for transfer learning
def create_transfer_model(num_classes, pretrained=True, freeze_features=False):
    # Create a transfer learning model
    # Args:
    #   num_classes: Number of output classes
    #   pretrained: Use pretrained weights
    #   freeze_features: Freeze pretrained layers (feature extraction mode)
    model = models.resnet18(pretrained=pretrained)
    
    # Freeze features if requested
    if freeze_features:
        for param in model.parameters():
            param.requires_grad = False
    
    # Replace classifier
    num_features = model.fc.in_features
    model.fc = nn.Linear(num_features, num_classes)
    
    return model

# Create model
model = create_transfer_model(num_classes=10, pretrained=True, freeze_features=False)
model = model.to(device)

# Loss and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.001, momentum=0.9)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=7, gamma=0.1)

print("Model setup complete!")


In [ ]:
# Training function
def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    for inputs, labels in loader:
        inputs, labels = inputs.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
    
    epoch_loss = running_loss / len(loader)
    epoch_acc = 100. * correct / total
    return epoch_loss, epoch_acc

def validate(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for inputs, labels in loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            
            running_loss += loss.item()
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
    
    epoch_loss = running_loss / len(loader)
    epoch_acc = 100. * correct / total
    return epoch_loss, epoch_acc

print("Training functions defined!")


In [ ]:
# Train the model
num_epochs = 5
history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

print("Starting training...")
print("="*70)

for epoch in range(num_epochs):
    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_acc = validate(model, val_loader, criterion, device)
    
    scheduler.step()
    
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    
    print(f"Epoch {epoch+1}/{num_epochs}:")
    print(f"  Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.2f}%")
    print(f"  Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.2f}%")
    print(f"  LR: {optimizer.param_groups[0]['lr']:.6f}")
    print()

print("Training complete!")


## 7. Comparing Different Architectures


In [ ]:
# Compare different pretrained models
architectures = {
    'ResNet18': models.resnet18,
    'ResNet50': models.resnet50,
    'VGG16': models.vgg16,
    'EfficientNet-B0': models.efficientnet_b0,
    'MobileNetV2': models.mobilenet_v2,
}

print("Model Comparison:")
print("="*70)
print(f"{'Model':<20} {'Parameters':<15} {'Input Size':<15}")
print("-"*70)

for name, model_fn in architectures.items():
    model = model_fn(pretrained=False)
    params = sum(p.numel() for p in model.parameters())
    print(f"{name:<20} {params:>12,}   224×224")

print()
print("Choosing the right model:")
print("  • ResNet18/34: Fast, good baseline")
print("  • ResNet50+: Higher accuracy, more parameters")
print("  • EfficientNet: Best accuracy/efficiency trade-off")
print("  • MobileNet: Fastest inference, mobile deployment")
print("  • VGG: Simple but large, good for visualization")


## 8. Advanced: Multi-Stage Fine-Tuning

Gradually unfreeze layers for better results.

```
Stage 1: Train only classifier (frozen backbone)
    ↓
Stage 2: Unfreeze last layer group
    ↓
Stage 3: Unfreeze more layers
    ↓
Stage 4: Fine-tune entire network

Gradual unfreezing prevents catastrophic forgetting!
```


In [ ]:
def set_trainable(model, layer_names, trainable=True):
    # Set specific layers to trainable or frozen
    for name, param in model.named_parameters():
        for layer_name in layer_names:
            if layer_name in name:
                param.requires_grad = trainable
                break

# Multi-stage fine-tuning example
model = models.resnet18(pretrained=True)
model.fc = nn.Linear(model.fc.in_features, 10)
model = model.to(device)

print("Multi-Stage Fine-Tuning Strategy:")
print("="*70)

# Stage 1: Freeze all, train classifier only
print("\nStage 1: Train classifier only")
for param in model.parameters():
    param.requires_grad = False
model.fc.weight.requires_grad = True
model.fc.bias.requires_grad = True
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"  Trainable parameters: {trainable:,}")

# Train for a few epochs...

# Stage 2: Unfreeze layer4
print("\nStage 2: Unfreeze layer4 + classifier")
set_trainable(model, ['layer4', 'fc'], trainable=True)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"  Trainable parameters: {trainable:,}")

# Train for a few more epochs...

# Stage 3: Unfreeze layer3
print("\nStage 3: Unfreeze layer3, layer4 + classifier")
set_trainable(model, ['layer3', 'layer4', 'fc'], trainable=True)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"  Trainable parameters: {trainable:,}")

# Stage 4: Fine-tune entire network
print("\nStage 4: Fine-tune entire network")
for param in model.parameters():
    param.requires_grad = True
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"  Trainable parameters: {trainable:,}")


## 9. Custom Classifier Heads

Replace the classifier with more complex architectures.


In [ ]:
# Advanced classifier head
class CustomClassifier(nn.Module):
    def __init__(self, num_features, num_classes, hidden_dim=512, dropout=0.5):
        super().__init__()
        self.classifier = nn.Sequential(
            nn.Linear(num_features, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, num_classes)
        )
    
    def forward(self, x):
        return self.classifier(x)

# Replace simple classifier with custom one
model = models.resnet18(pretrained=True)
num_features = model.fc.in_features
model.fc = CustomClassifier(num_features, num_classes=10)
model = model.to(device)

print("Model with custom classifier:")
print(model.fc)
print()

# Count new classifier parameters
classifier_params = sum(p.numel() for p in model.fc.parameters())
print(f"Custom classifier parameters: {classifier_params:,}")


## 10. Best Practices for Transfer Learning


In [ ]:
print("Transfer Learning Best Practices:")
print("="*70)
print()

print("1. DATA PREPROCESSING")
print("   ✓ Use same normalization as pretrained model (ImageNet)")
print("   ✓ Input size should match expected dimensions")
print("   ✓ Apply appropriate data augmentation")
print()

print("2. LEARNING RATE")
print("   ✓ Feature extraction: Use normal LR (1e-3)")
print("   ✓ Fine-tuning: Use 10-100x smaller LR (1e-4 to 1e-5)")
print("   ✓ Different LR for pretrained vs new layers")
print()

print("3. TRAINING STRATEGY")
print("   ✓ Start with feature extraction")
print("   ✓ Then fine-tune gradually (unfreeze layer by layer)")
print("   ✓ Use smaller batch sizes if memory limited")
print()

print("4. WHEN TO USE WHICH")
print("   • Small dataset (<1K samples): Feature extraction only")
print("   • Medium dataset (1K-10K): Feature extraction → Fine-tuning")
print("   • Large dataset (>10K): Full fine-tuning")
print("   • Very different domain: More fine-tuning needed")
print()

print("5. ARCHITECTURE SELECTION")
print("   • ResNet18/34: Quick experiments, limited compute")
print("   • ResNet50: Good balance")
print("   • EfficientNet: Best accuracy/speed trade-off")
print("   • MobileNet: Mobile/edge deployment")
print()

print("6. COMMON MISTAKES TO AVOID")
print("   ✗ Forgetting to normalize inputs correctly")
print("   ✗ Using same LR for all layers when fine-tuning")
print("   ✗ Not using data augmentation")
print("   ✗ Training too long on small datasets (overfitting)")
print("   ✗ Not validating on a held-out set")


## 📝 Summary

✅ Loaded and used pretrained models  
✅ Implemented feature extraction strategy  
✅ Applied fine-tuning techniques  
✅ Used proper data preprocessing  
✅ Compared different architectures  
✅ Implemented multi-stage fine-tuning  
✅ Created custom classifier heads  
✅ Learned best practices  

### Key Takeaways

1. **Transfer learning is powerful** - Use it whenever possible!
2. **Feature extraction first** - Start frozen, then fine-tune
3. **Lower LR for pretrained layers** - Preserve learned features
4. **Correct preprocessing is critical** - Match pretrained model expectations
5. **Gradual unfreezing works best** - Don't unfreeze everything at once

### When Transfer Learning Works Best

- ✅ Limited training data
- ✅ Similar domain to pretraining (e.g., ImageNet → other images)
- ✅ Want faster training
- ✅ Need better baseline performance

### Next Steps

- **Next**: `05_evaluation.ipynb` - Evaluate your models properly
- **Practice**: Transfer learn on your own dataset
- **Challenge**: Compare feature extraction vs fine-tuning on same dataset


## 🎯 Practice Exercises


In [ ]:
# Exercise 1: Load EfficientNet and prepare for 5-class classification
# Your code here:


# Exercise 2: Implement discriminative learning rates (different LR per layer)
# Your code here:


# Exercise 3: Create a custom multi-head classifier
# Your code here:


# Exercise 4: Compare training from scratch vs transfer learning
# Your code here:


# Exercise 5: Implement progressive unfreezing during training
# Your code here:
